In [1]:
# Parameters
run_id = "130b4b0c-059c-4a33-880b-6e04c41ce18c"
artifacts_dir = "/home/adnoman/projects/aml_gan/AMLend2end/artifacts/runs/130b4b0c-059c-4a33-880b-6e04c41ce18c"
sample_size = None
epochs = None
threshold = None


### Train adversarial anomaly detection model
In the previous notebook we performed hyperparamer tuning for adversarial anomaly detection model. Now we are ready to train the model based on the best hyper parameters and export to model repository.
![Training Dataset](./images/experiment_td.png)

In [2]:
# Setup for local execution
import os
import json
import uuid
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score, classification_report
import matplotlib.pyplot as plt

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
RESOURCES_PATH = os.path.join(BASE_PATH, "Resources")
MODELS_PATH = os.path.join(BASE_PATH, "models")
GAN_DATA_PATH = os.path.join(TRAINING_DATA_PATH, "gan")

print(f"TensorFlow version: {tf.__version__}")

2026-02-02 16:20:51.334089: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 16:20:51.719785: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-02-02 16:20:53.230692: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0


## Connect to hsfs and retrieve datasets for training and evaluation 

In [3]:
# Load hyperparameters
emb_hp_path = os.path.join(RESOURCES_PATH, "embeddings_best_hp.json")
with open(emb_hp_path, 'r') as f:
    emb_best_hp = json.load(f)

gan_hp_path = os.path.join(RESOURCES_PATH, "gan_best_hp.json")
with open(gan_hp_path, 'r') as f:
    gan_best_hp = json.load(f)

input_dim = emb_best_hp['emb_size']
print(f"Embedding hyperparameters: {emb_best_hp}")
print(f"GAN hyperparameters: {gan_best_hp}")

Embedding hyperparameters: {'walk_number': 2, 'walk_length': 2, 'emb_size': 32}
GAN hyperparameters: {'latent_dim': 8, 'n_layers': 2, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.0001}


### Define hopsworks experiments wrapper function and put all the training logic there. 

In [4]:
# Load training data
X_train = np.load(os.path.join(GAN_DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(GAN_DATA_PATH, "y_train.npy"))
X_eval = np.load(os.path.join(GAN_DATA_PATH, "X_eval.npy"))
y_eval = np.load(os.path.join(GAN_DATA_PATH, "y_eval.npy"))

print(f"Training data: {X_train.shape}")
print(f"Evaluation data: {X_eval.shape}")
print(f"Evaluation labels - SAR: {y_eval.sum()}, Non-SAR: {(y_eval==0).sum()}")

Training data: (5224, 32)
Evaluation data: (2123, 32)
Evaluation labels - SAR: 816, Non-SAR: 1307


## Use above experiments wrapper function to conduct hops training experiments.

In [5]:
# Build autoencoder with best hyperparameters
def build_autoencoder(input_dim, latent_dim, n_layers, activation, dropout_rate, learning_rate):
    """Build an autoencoder for anomaly detection."""
    
    # Encoder
    encoder_input = layers.Input(shape=(input_dim,))
    x = encoder_input
    
    units = input_dim
    for i in range(n_layers):
        units = max(units // 2, latent_dim)
        x = layers.Dense(units, activation=activation)(x)
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate)(x)
    
    latent = layers.Dense(latent_dim, activation=activation, name='latent')(x)
    
    # Decoder
    x = latent
    units = latent_dim
    for i in range(n_layers):
        units = min(units * 2, input_dim)
        x = layers.Dense(units, activation=activation)(x)
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate)(x)
    
    decoder_output = layers.Dense(input_dim, activation='linear')(x)
    
    # Full autoencoder
    autoencoder = keras.Model(encoder_input, decoder_output, name='autoencoder')
    autoencoder.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse'
    )
    
    return autoencoder

# Build model
model = build_autoencoder(
    input_dim=input_dim,
    latent_dim=gan_best_hp['latent_dim'],
    n_layers=gan_best_hp['n_layers'],
    activation=gan_best_hp['activation'],
    dropout_rate=gan_best_hp['dropout_rate'],
    learning_rate=gan_best_hp['learning_rate']
)

model.summary()

I0000 00:00:1770031254.481718   87250 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9511 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,480 (9.69 KB)

 Trainable params: 2,480 (9.69 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Train the model
EPOCHS = 50
BATCH_SIZE = 32

print("Training anomaly detection model...")
history = model.fit(
    X_train, X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1
)

print("\nTraining complete!")

Training anomaly detection model...
Epoch 1/50


2026-02-02 16:20:55.849822: I external/local_xla/xla/service/service.cc:163] XLA service 0x741e38005200 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-02 16:20:55.849867: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9
2026-02-02 16:20:55.878204: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-02 16:20:56.026026: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801


  1/147 ━━━━━━━━━━━━━━━━━━━━ 4:10 2s/step - loss: 3.5796e-04

 35/147 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.3569e-04 

 73/147 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.3190e-04

...

 71/147 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.7518e-04

104/147 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.7503e-04

139/147 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.7494e-04

147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.7429e-04 - val_loss: 2.7415e-04



Training complete!


In [7]:
# Evaluate the model
def compute_anomaly_score(model, X):
    """Compute reconstruction error as anomaly score."""
    X_pred = model.predict(X, verbose=0)
    mse = np.mean(np.square(X - X_pred), axis=1)
    return mse

# Compute anomaly scores
anomaly_scores = compute_anomaly_score(model, X_eval)

# Calculate AUC
auc = roc_auc_score(y_eval, anomaly_scores)
print(f"Anomaly Detection AUC: {auc:.4f}")

# Find optimal threshold
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_eval, anomaly_scores)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal threshold: {optimal_threshold:.6f}")

Anomaly Detection AUC: 0.4913
Optimal threshold: 0.000119


In [8]:
# Classification report
y_pred = (anomaly_scores > optimal_threshold).astype(int)
print("\nClassification Report:")
print(classification_report(y_eval, y_pred, target_names=['Non-SAR', 'SAR']))


Classification Report:
              precision    recall  f1-score   support

     Non-SAR       0.50      0.00      0.00      1307
         SAR       0.38      1.00      0.55       816

    accuracy                           0.38      2123
   macro avg       0.44      0.50      0.28      2123
weighted avg       0.46      0.38      0.21      2123



In [9]:
# Save the model locally (replaces Hopsworks model registry)
model_id = str(uuid.uuid4())[:8]
model_dir = os.path.join(MODELS_PATH, f"gan_anomaly_{model_id}")
os.makedirs(model_dir, exist_ok=True)

# Save Keras model
model_path = os.path.join(model_dir, "anomaly_detector.keras")
model.save(model_path)
print(f"Saved model to: {model_path}")

# Save metadata
metadata = {
    'hyperparameters': gan_best_hp,
    'embedding_dim': input_dim,
    'metrics': {
        'auc': float(auc),
        'optimal_threshold': float(optimal_threshold),
        'final_loss': float(history.history['loss'][-1])
    }
}
metadata_path = os.path.join(model_dir, "metadata.json")
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata to: {metadata_path}")

# Save threshold for inference
threshold_path = os.path.join(model_dir, "threshold.npy")
np.save(threshold_path, optimal_threshold)

print(f"\n{'='*50}")
print(f"Model saved to: {model_dir}")
print(f"AUC: {auc:.4f}")
print(f"{'='*50}")

Saved model to: /home/adnoman/projects/aml_gan/AMLend2end/models/gan_anomaly_30a33f36/anomaly_detector.keras
Saved metadata to: /home/adnoman/projects/aml_gan/AMLend2end/models/gan_anomaly_30a33f36/metadata.json

Model saved to: /home/adnoman/projects/aml_gan/AMLend2end/models/gan_anomaly_30a33f36
AUC: 0.4913


### Managing experiments
Experiments service provides a unified view of all the experiments run using the `experiment` module.
<br>
As demonstrated in the gif it provides general information about the experiment and the resulting metric. Experiments can be visualized meanwhile or after training in a TensorBoard.
<br>
<br>
![Image7-Monitor.png](./images/experiments.gif)